# Xiaomi-OCR-0 — document OCR, KIE, and VQA on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/melvin/xiaomi-ocr-0-document-ocr-7143/colab/Xiaomi-OCR-0-document-ocr.ipynb)

[`xiaomi-research/Xiaomi-OCR-0`](https://huggingface.co/xiaomi-research/Xiaomi-OCR-0) is Xiaomi / SeerRay's **0.8B** OCR-specialized vision-language model (Apache 2.0, not gated). It is a Qwen3.5-0.8B-Base fine-tune (`Qwen3_5ForConditionalGeneration`, ~873M parameters, about **1.7 GB** of weights). The Hub pipeline tag is `image-text-to-text`. The same checkpoint is mirrored at [`SeerRay-Lab/Xiaomi-OCR-0`](https://huggingface.co/SeerRay-Lab/Xiaomi-OCR-0).

Why this model: **one compact checkpoint** covers three jobs that usually need different models — whole-page **document parsing** (Markdown, tables as OTSL, formulas as LaTeX), **key information extraction** to JSON, and **OCR-oriented visual question answering**. The card reports **95.24** on Real5-OmniDocBench (#1 in their comparison) and a mean **83.2** across five OCR-VQA benches, above Qwen3.5-2B (80.9) at 0.8B.

**Licence:** Apache 2.0.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has about **15 GB** of VRAM, **no native bfloat16**, and **no FlashAttention 2**. The card's verified install pins `transformers==5.17.0` and loads **float16** on CUDA (`torch.float32` only on CPU). This notebook keeps that dtype and uses **SDPA** (eager if SDPA refuses Qwen3.5's hybrid / linear-attention stack). Weights are about 1.7 GB; a receipt or page whose longest edge is 2048 px should peak around **5–8 GB**. That estimate is from the weight file plus a 2048 px page, not a measured Colab run.

**Not in this notebook:** SGLang / vLLM serving, the two-stage PP-DocLayoutV3 region pipeline (PaddlePaddle), the MCP Skill / browser demo, Ollama GGUF, or the card's `torch==2.14.0` pin. Colab's installed torch is used as-is. Do not install `flash-attn`.

**Links:** [model card](https://huggingface.co/xiaomi-research/Xiaomi-OCR-0) · [mirror](https://huggingface.co/SeerRay-Lab/Xiaomi-OCR-0) · [GitHub](https://github.com/SeerRay-Lab/Xiaomi-OCR-0) · [paper](https://arxiv.org/abs/2609.36136) · [demo](https://huggingface.co/spaces/SeerRay-Lab/Xiaomi-OCR-0)

In [ ]:
%pip install -q "transformers==5.17.0" pillow

# The card's verified combo is transformers==5.17.0, torch==2.14.0,
# torchvision==0.29.0, pillow. This notebook pins only transformers and
# leaves Colab's torch / torchvision alone (do not force torch==2.14).
# Use %pip, not `python -I -m pip`: isolated mode drops Colab site-packages.
# Do NOT install flash-attn. Qwen3.5 linear-attention uses the reference
# PyTorch path on a T4 (same output, slower, no CUDA kernel build).
# pillow is already on Colab; listed because the card lists it.

> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The pin to 5.17.0 is invisible until the process restarts.

In [ ]:
import torch
import transformers
from packaging.version import Version

print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

assert Version(transformers.__version__) >= Version("5.17.0"), (
    f"transformers {transformers.__version__} is loaded; Xiaomi-OCR-0's card "
    "verifies 5.17.0. Run the install cell, then Runtime > Restart session."
)

# T4 (sm_75) has no FlashAttention 2 and no native bfloat16.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

major, minor = torch.cuda.get_device_capability()
# Card: float32 on CPU, float16 on CUDA. T4 has no bf16 tensor cores.
DTYPE = torch.float16 if major < 8 else torch.bfloat16
print("device:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
free, total = torch.cuda.mem_get_info()
print(f"VRAM free: {free / 1024**3:.1f} GB of {total / 1024**3:.1f} GB")
print("dtype:", DTYPE)
print("transformers", transformers.__version__, "| torch", torch.__version__)
if major < 8:
    print("T4 path: float16 + SDPA/eager. bfloat16 and FlashAttention-2 are off.")


def print_vram(label=""):
    suffix = f"  ({label})" if label else ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"VRAM allocated: {allocated:.2f} GB | peak: {peak:.2f} GB{suffix}")

## 1. Load Xiaomi-OCR-0

The card loads `AutoProcessor` + `AutoModelForImageTextToText` (resolves to `Qwen3_5ForConditionalGeneration`) and does **not** set `trust_remote_code`. Thinking stays off: the evaluation pipeline and Ollama notes both pass `enable_thinking=False`; with it on, OCR output can be empty or padded with `<think>` tokens. Sampling is greedy (`do_sample=False`), matching the card. The first run downloads about 1.7 GB.

Official eval resizes pages to a **2048 × 2048** bound. The processor's default `longest_edge` is huge, so this notebook also thumbnail-caps the image before it reaches the model.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "xiaomi-research/Xiaomi-OCR-0"  # mirror: SeerRay-Lab/Xiaomi-OCR-0


def load_ocr(attn):
    return AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        dtype=DTYPE,
        attn_implementation=attn,
    ).to("cuda").eval()


torch.cuda.reset_peak_memory_stats()
# SDPA is the T4 path. Fall back to eager if this transformers build rejects SDPA
# on Qwen3.5's linear-attention layers.
try:
    model = load_ocr("sdpa")
    ATTN = "sdpa"
except Exception as exc:
    print("sdpa load failed, retrying eager:", exc)
    model = load_ocr("eager")
    ATTN = "eager"

processor = AutoProcessor.from_pretrained(MODEL_ID)
n_params = sum(p.numel() for p in model.parameters())
print("class:", type(model).__name__)
print(f"parameters: {n_params / 1e9:.2f}B | dtype: {model.dtype} | attention: {ATTN}")
print_vram("after load")

## 2. Helpers, official prompts, and a sample receipt

`ask(image, prompt)` is the card's Transformers path: chat template with `enable_thinking=False`, then greedy `generate`. Prompts are copied from the [model card](https://huggingface.co/xiaomi-research/Xiaomi-OCR-0) / [evaluation pipeline](https://github.com/SeerRay-Lab/Xiaomi-OCR-0/blob/main/evaluation_pipeline/e2e_img2md.py).

Sample: the official `example_pics/receipt.jpg` from the checkpoint repo (SROIE-style MR. D.I.Y. ticket — the same image as the card's KIE example). If the Hub fetch fails, the notebook renders a tiny invoice with PIL so the three demos still run.

In [ ]:
import io
import json
import re
import time

import requests
from huggingface_hub import hf_hub_download
from IPython.display import Markdown, display
from PIL import Image, ImageDraw, ImageFont

LONGEST = 2048  # official eval bound (fit_image → 2048 × 2048)
UA = {"User-Agent": "donvito-notebooks/1.0 (educational Colab demo)"}

DOCUMENT_PROMPT = (
    "Extract all information from the main body of the document image and represent it "
    "in markdown format, ignoring headers and footers. Tables should be expressed in OTSL "
    "format, formulas in the document should be represented using LATEX format, and the "
    "parsing should be organized according to the reading order."
)
KIE_PROMPT = """Extract key information in the image

Please output the key information in JSON format according to the following schema:
{
    "date": "",
    "seller_address": "",
    "seller_gst_id": "",
    "seller_name": "",
    "total_amount": "",
    "total_tax": ""
}
"""
VQA_PROMPT = "What is the seller name, the receipt date, and the total amount?"


def limit_edge(image, longest=LONGEST):
    image = image.convert("RGB")
    image.thumbnail((longest, longest))
    return image


def render_sample_invoice():
    image = Image.new("RGB", (720, 900), "white")
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()
    lines = [
        "ACME OFFICE SUPPLY SDN BHD",
        "LOT 12, JALAN INDUSTRI 3,",
        "43300 SERI KEMBANGAN, SELANGOR",
        "GST ID: 001092886528",
        "",
        "TAX INVOICE                    2017-10-18",
        "",
        "A4 paper ream           2 x 2.50     5.00",
        "Blue ballpoint pen      3 x 0.30     0.90",
        "",
        "SUBTOTAL                             5.90",
        "GST 6%                               0.33",
        "TOTAL                                5.90",
        "",
        "Thank you. Please come again.",
    ]
    y = 40
    for line in lines:
        draw.text((40, y), line, fill="black", font=font)
        y += 36
    return image


def fetch_receipt():
    urls = [
        "https://huggingface.co/xiaomi-research/Xiaomi-OCR-0/resolve/main/example_pics/receipt.jpg",
        "https://huggingface.co/SeerRay-Lab/Xiaomi-OCR-0/resolve/main/example_pics/receipt.jpg",
        "https://huggingface.co/datasets/hf-internal-testing/fixtures_ocr/resolve/main/SROIE-receipt.jpeg",
    ]
    try:
        path = hf_hub_download(MODEL_ID, filename="example_pics/receipt.jpg")
        return limit_edge(Image.open(path)), path
    except Exception as hub_exc:
        print("hf_hub_download failed, trying HTTP mirrors:", hub_exc)
    last = None
    for url in urls:
        try:
            response = requests.get(url, headers=UA, timeout=60)
            response.raise_for_status()
            return limit_edge(Image.open(io.BytesIO(response.content))), url
        except Exception as exc:
            last = exc
    print("public receipt fetch failed, rendering a PIL invoice:", last)
    return render_sample_invoice(), "rendered:pil-invoice"


def to_model(inputs):
    inputs = inputs.to(model.device)
    for key, value in list(inputs.items()):
        if hasattr(value, "is_floating_point") and value.is_floating_point():
            inputs[key] = value.to(dtype=model.dtype)
    return inputs


def ask(image, prompt, max_new_tokens=1024):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    )
    inputs = to_model(inputs)
    start = time.time()
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = output[:, inputs["input_ids"].shape[1]:]
    text = processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()
    print(f"[{new_tokens.shape[1]} tokens in {time.time() - start:.1f}s]")
    print_vram()
    return text


def otsl_to_markdown(otsl):
    body = otsl.replace("<otsl>", "").replace("</otsl>", "")
    rows = []
    for raw_row in body.split("<nl>"):
        if not raw_row.strip():
            continue
        cells = []
        for tag, value in re.findall(r"<(fcel|ecel|lcel|ucel|xcel)>([^<]*)", raw_row):
            cells.append(value.strip() if tag == "fcel" else "")
        if cells:
            rows.append(cells)
    if not rows:
        return otsl
    width = max(len(r) for r in rows)
    rows = [r + [""] * (width - len(r)) for r in rows]
    lines = ["| " + " | ".join(rows[0]) + " |", "|" + "---|" * width]
    lines += ["| " + " | ".join(r) + " |" for r in rows[1:]]
    return "\n".join(lines)


receipt, receipt_src = fetch_receipt()
print("source:", receipt_src)
print("receipt", receipt.size)

### Demo 1 — parse the receipt to Markdown

Whole-page document prompt from the card. Tables, if any, come back as OTSL tags; `otsl_to_markdown` turns those into a Markdown grid (the official postprocessor emits HTML instead).

In [ ]:
display(receipt)
parsed = ask(receipt, DOCUMENT_PROMPT, max_new_tokens=1024)
print(parsed)
if "<fcel>" in parsed or "<otsl>" in parsed:
    display(Markdown(otsl_to_markdown(parsed)))

### Demo 2 — key information extraction to JSON

The card's KIE prompt: `Extract key information in the image` plus a JSON schema. This is the official receipt schema (`seller_name` / date / `total_amount`, plus address, GST id, and tax). Always parse the JSON before you trust it downstream.

In [ ]:
kie_raw = ask(receipt, KIE_PROMPT, max_new_tokens=256)
print(kie_raw)

match = re.search(r"\{.*\}", kie_raw, re.S)
try:
    fields = json.loads(match.group(0)) if match else {}
except json.JSONDecodeError as exc:
    print("JSON parse failed (model can emit malformed JSON):", exc)
    fields = {}
print("\nparsed:")
print(json.dumps(fields, indent=2, ensure_ascii=False))
for key in ("seller_name", "date", "total_amount", "seller_gst_id", "total_tax"):
    print(f"{key:>16}: {fields.get(key, '<missing>')}")

### Demo 3 — document VQA

OCR-centric visual question answering: the image plus a question, no extra task prefix. The card does not publish a special VQA wrapper for the Transformers path (the MCP Skill adds a concise-answer format for `vqa_image`). Here the question is asked directly.

In [ ]:
print("question:", VQA_PROMPT)
vqa = ask(receipt, VQA_PROMPT, max_new_tokens=128)
print(vqa)

### Peak VRAM after the three demos

In [ ]:
print("document parse (first 400 chars):")
print((parsed or "")[:400])
print("\nKIE JSON:")
print(json.dumps(fields, indent=2, ensure_ascii=False) if fields else kie_raw)
print("\nVQA:")
print(vqa)
print()
print_vram("after 3 demos")
free, total = torch.cuda.mem_get_info()
print(f"VRAM free: {free / 1024**3:.1f} GB of {total / 1024**3:.1f} GB")

## 3. Caveats and cleanup

**What this notebook is for.** The official Transformers path on a free T4: `AutoProcessor` + `AutoModelForImageTextToText`, the card's document / KIE prompts, and a short document VQA question. Use it to see the raw Markdown / JSON / answer and allocated / peak VRAM.

**What it is not.** Production serving on the card is SGLang or vLLM. Full-page printed documents can go through a two-stage pipeline (PP-DocLayoutV3 finds regions; the model reads each crop; postprocess converts OTSL to HTML). That needs PaddlePaddle and is out of scope here. The MCP Skill / browser demo apply extra image preprocessing; this notebook is the published Transformers snippet plus a 2048 px cap.

| Task | Prompt | Output |
| --- | --- | --- |
| Document | `Extract all information from the main body...` (card) | Markdown; tables as OTSL, formulas as LaTeX |
| Text region | `Extract the text in the image.` | Plain text |
| Table region | `Parse the table in the image into OTSL.` | OTSL |
| Formula region | `Identify the formula in the image and represent it using LATEX format.` | LaTeX |
| Key information extraction | `Extract key information in the image` + JSON schema | JSON object |
| Document VQA | Your question, no extra prefix | Short answer |

**T4 limits.** float16 (no native bfloat16), SDPA or eager (no FlashAttention 2), and the reference PyTorch path for linear-attention layers. Images are capped at a 2048 px longest edge. The card's `max_new_tokens=4096` is for dense pages; receipts need far less. Keep `enable_thinking=False`.

**Related notebooks:** [MiniCPM-OCR 1.0](MiniCPM-OCR-1.0-document-ocr.ipynb) (region prompts + KIE), [Unlimited-OCR-2](Unlimited-OCR-2-document-ocr.ipynb) (page transcription), [LightOnOCR-3](LightOnOCR-3-0.8B-document-ocr.ipynb) (transcription + boxes), [OvisOCR3](OvisOCR3-structured-json.ipynb) (schema JSON), [Indra OCR](Indra-OCR-1M-document-ocr.ipynb) (CPU).

Free the GPU before you open another notebook:

In [ ]:
import gc

print_vram("before cleanup")
del model, processor
gc.collect()
torch.cuda.empty_cache()
print_vram("after cleanup")